In [40]:
import os
import numpy as np
import pandas as pd
import nibabel as nib
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
import torch
import random

In [13]:
os.chdir('/home/ntdung/Medical')

In [39]:
excel_path = 'data/participants.xlsx'
df = pd.read_excel(excel_path, nrows=None)
df

,No.,subject_age,subject_dx,subject_sex,subject_id,dataset_name
0,1,44.2,control,m,sub-BrainAge000019,ABIDE/Caltech
1,2,39.3,control,m,sub-BrainAge000020,ABIDE/Caltech
2,3,42.5,control,m,sub-BrainAge000021,ABIDE/Caltech
3,4,19.7,control,m,sub-BrainAge000022,ABIDE/Caltech
4,5,20.0,control,f,sub-BrainAge000023,ABIDE/Caltech
...,...,...,...,...,...,...
4943,4944,66.0,control,f,sub-BrainAge023209,RocklandSample
4944,4945,69.0,control,m,sub-BrainAge023210,RocklandSample
4945,4946,23.0,control,m,sub-BrainAge023211,RocklandSample
4946,4947,54.0,control,f,sub-BrainAge023212,RocklandSample


In [46]:
def is_integer(n):
    return float(n).is_integer()

n_total = len(df)
n_integer = df['subject_age'].apply(is_integer).sum()
n_decimal = n_total - n_integer

print(f"Samples with integer age values: {n_integer}")
print(f"Samples with decimal age values: {n_decimal}")

Samples with integer age values: 4242
Samples with decimal age values: 706


In [47]:
df['subject_age'] = df['subject_age'].round().astype(int)
df

,No.,subject_age,subject_dx,subject_sex,subject_id,dataset_name
0,1,44,control,m,sub-BrainAge000019,ABIDE/Caltech
1,2,39,control,m,sub-BrainAge000020,ABIDE/Caltech
2,3,42,control,m,sub-BrainAge000021,ABIDE/Caltech
3,4,20,control,m,sub-BrainAge000022,ABIDE/Caltech
4,5,20,control,f,sub-BrainAge000023,ABIDE/Caltech
...,...,...,...,...,...,...
4943,4944,66,control,f,sub-BrainAge023209,RocklandSample
4944,4945,69,control,m,sub-BrainAge023210,RocklandSample
4945,4946,23,control,m,sub-BrainAge023211,RocklandSample
4946,4947,54,control,f,sub-BrainAge023212,RocklandSample


In [49]:
def load_middle_slices(nii_path):
    img = nib.load(nii_path).get_fdata()

    x, y, z = img.shape
    axial_slice = img[:, :, z // 2]
    sagittal_slice = img[x // 2, :, :]
    coronal_slice = img[:, y // 2, :]

    # Normalize
    axial_slice = (axial_slice - axial_slice.min()) / (axial_slice.max() - axial_slice.min())
    sagittal_slice = (sagittal_slice - sagittal_slice.min()) / (sagittal_slice.max() - sagittal_slice.min())
    coronal_slice = (coronal_slice - coronal_slice.min()) / (coronal_slice.max() - coronal_slice.min())

    return axial_slice, sagittal_slice, coronal_slice

In [50]:
class BrainMRIDataset(Dataset):
    def __init__(self, dataframe, root_dir):
        self.df = dataframe
        self.root_dir = root_dir

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        subject_id = row['subject_id']
        age = row['subject_age']
        sex = 0 if row['subject_sex'] == 'm' else 1

        nii_path = os.path.join(self.root_dir, subject_id, 'anat', f'{subject_id}_T1w.nii.gz')
        axial, sagittal, coronal = load_middle_slices(nii_path)

        # Stack to tensor (3, H, W)
        image = np.stack([axial, sagittal, coronal], axis=0).astype(np.float32)

        while True:
            cf_idx = random.randint(0, len(self.df) - 1)
            if cf_idx != idx:
                break
        cf_row = self.df.iloc[cf_idx]
        cf_age = cf_row['subject_age']
        cf_sex = 0 if cf_row['subject_sex'] == 'm' else 1

        return {
            'image': torch.tensor(image),
            'age': torch.tensor(age, dtype=torch.float32),
            'sex': torch.tensor(sex, dtype=torch.long),
            'cf_age': torch.tensor(cf_age, dtype=torch.float32),
            'cf_sex': torch.tensor(cf_sex, dtype=torch.long),
            'subject_id': subject_id
        }


In [56]:
dataset = BrainMRIDataset(df, 'data')
dataloader = DataLoader(dataset, batch_size=4, shuffle=True)

In [57]:
batch = next(iter(dataloader))
print("Image shape:", batch['image'].shape)  # (B, 3, H, W)
print("Age:", batch['age'])
print("Sex:", batch['sex'])
print("CF Age:", batch['cf_age'])
print("CF Sex:", batch['cf_sex'])
print("Subject IDs:", batch['subject_id'])

Image shape: torch.Size([4, 3, 130, 130])
Age: tensor([59., 20., 58., 49.])
Sex: tensor([1, 0, 1, 0])
CF Age: tensor([25., 62., 21., 60.])
CF Sex: tensor([1, 0, 1, 1])
Subject IDs: ['sub-BrainAge019847', 'sub-BrainAge019478', 'sub-BrainAge022412', 'sub-BrainAge022724']
